# Primer trim — MaskPrimers (pRESTO)

Только для **human (PRJEB40348)** и **horse (PRJNA848968)**.
Macaque и sheep — 5'RACE, не требуют.

Вход: trimmed FASTQ из `results/<DS>/trimmed/fastq/`.
Выход: `results/<DS>/pr_trimmed/{fastq, maskprimer_logs}/`.

**Без fastp, без фильтрации** — только вырезание V-праймера.


In [ ]:
# PATH к conda-окружению на OneQ VM (bcr_env).
# Если kernel уже зарегистрирован как "BCR Pipeline" — эта ячейка не нужна,
# но оставляем для надёжности (после Restart Kernel обязательно выполнить).
import os, sys
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
if os.path.isdir(_CONDA_ENV + "/lib/python3.11/site-packages"):
    sys.path.insert(0, _CONDA_ENV + "/lib/python3.11/site-packages")


In [ ]:
import subprocess
from pathlib import Path

PRIMERS = {
    "PRJEB40348":  "primer_refs/human_primers.fasta",
    "PRJNA848968": "primer_refs/horse_primers.fasta",
}


In [ ]:
def run_primer_trim(volume, dataset):
    if dataset not in PRIMERS:
        print(f"[primer_trim] {dataset}: 5'RACE, skip")
        return

    primers = PRIMERS[dataset]
    vol = Path(volume)
    src = vol / "results" / dataset / "trimmed" / "fastq"
    base = vol / "results" / dataset / "pr_trimmed"

    out = base / "fastq"
    logs = base / "maskprimer_logs"
    out.mkdir(parents=True, exist_ok=True)
    logs.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(
        f.name.replace("_1.trim.fastq.gz", "")
        for f in src.glob("*_1.trim.fastq.gz")
    ))
    print(f"[primer_trim] {dataset}: {len(pairs)} pairs")

    for bn in pairs:
        r1 = src / f"{bn}_1.trim.fastq.gz"
        if not r1.exists():
            continue

        print(f"  {bn} R1 ...")
        subprocess.run(["MaskPrimers.py", "align",
                        "-s", str(r1), "-p", primers,
                        "--mode", "cut", "--maxerror", "0.2",
                        "--nproc", "4", "--maxlen", "50",
                        "--outdir", str(out), "--outname", f"{bn}_1.pr"],
                       capture_output=True)

        print(f"  {bn} R2 ...")
        subprocess.run(["MaskPrimers.py", "align",
                        "-s", str(src / f"{bn}_2.trim.fastq.gz"),
                        "-p", primers,
                        "--mode", "cut", "--maxerror", "0.2",
                        "--nproc", "4", "--maxlen", "50",
                        "--outdir", str(out), "--outname", f"{bn}_2.pr"],
                       capture_output=True)

        for f in out.glob(f"{bn}_*.pr_primers-pass.fastq.gz"):
            f.rename(out / f.name.replace("_primers-pass", ""))
        print(f"  {bn} done")

    print(f"[primer_trim] DONE")


### Запуск

Только для human и horse. Для остальных — `skip`.


In [ ]:
run_primer_trim("/data/user/epishkin", "PRJEB40348")


In [ ]:
run_primer_trim("/data/user/epishkin", "PRJNA848968")
